# model.train() と model.eval() の違いを実測する

**出典**　機械学習 4 分野の教科書 — 第 6 章 2-A 基礎と学習の安定化 ／ 07 演習と解答

**このノートのパス**　`ml-code/06_nn_basics/01_train_eval_mode.ipynb`

**このノートで確かめること**
教科書の断片を動く形に補う。**同じ入力・同じ重みでも、train モードと eval モードで
出力が変わる**ことを Dropout と BatchNorm それぞれで数値で確かめる。
`eval()` の付け忘れがなぜ静かなバグになるのかを体感する。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `torch` |
| 実行時間の目安 | 約 10 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q torch

In [3]:
import torch
import torch.nn as nn

torch.manual_seed(0)

# Dropout と BatchNorm を含む、ごく小さなモデル
model = nn.Sequential(
    nn.Linear(10, 32),
    nn.BatchNorm1d(32),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(32, 2),
)

x = torch.randn(16, 10)          # バッチ 16
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

device: cpu


## ① 同じ入力を train モードで 3 回通す — 毎回答えが変わる

In [4]:
model.train()      # 学習時：dropout 有効、BatchNorm はバッチ統計を使う
for i in range(3):
    out = model(x)
    print(f"train 回目{i+1}: {out[0].detach().numpy().round(4)}")

train 回目1: [0.388  0.4757]
train 回目2: [0.8788 0.0291]
train 回目3: [-0.7255  0.6828]


## ② eval モードなら何度通しても同じ

In [5]:
model.eval()       # 推論時：dropout 無効、BatchNorm は移動平均を使う
with torch.no_grad():
    for i in range(3):
        out = model(x)
        print(f"eval  回目{i+1}: {out[0].numpy().round(4)}")

eval  回目1: [0.3486 0.1633]
eval  回目2: [0.3486 0.1633]
eval  回目3: [0.3486 0.1633]


## ③ BatchNorm は「バッチの中身」に依存する — バッチサイズ 1 で壊れる

train モードのまま 1 件だけ流すと、そのバッチ 1 件の平均・分散で正規化することになる。
分散が定義できないので、**PyTorch は例外を投げる**（静かに壊れないのはありがたい）。

ところが **eval モードなら 1 件でも通る**。移動平均を使うからである。

In [6]:
one = x[:1]

model.train()
try:
    with torch.no_grad():
        print("train モードで 1 件:", model(one)[0].numpy().round(4))
except ValueError as e:
    print("train モードで 1 件 → 例外:", e)

model.eval()
with torch.no_grad():
    print("eval  モードで 1 件  :", model(one)[0].numpy().round(4))
    print("eval  モードでバッチごと:", model(x)[0].numpy().round(4),
          "← 1 件のときと一致する")

train モードで 1 件 → 例外: Expected more than 1 value per channel when training, got input size torch.Size([1, 32])
eval  モードで 1 件  : [0.3486 0.1633]
eval  モードでバッチごと: [0.3486 0.1633] ← 1 件のときと一致する


## ④ 学習ループでの正しい置き場所

教科書の断片は、実際の学習ループではこう配置する。

In [7]:
from torch.utils.data import TensorDataset, DataLoader

ds = TensorDataset(torch.randn(256, 10), torch.randint(0, 2, (256,)))
train_loader = DataLoader(ds, batch_size=32, shuffle=True)
x_test = torch.randn(64, 10)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(2):
    model.train()      # ⚠️ 毎エポック頭で必ず呼ぶ
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()

    model.eval()       # ⚠️ 推論の前に必ず呼ぶ
    with torch.no_grad():
        preds = model(x_test)
    print(f"epoch {epoch}: loss={loss.item():.4f} preds={tuple(preds.shape)}")

epoch 0: loss=0.7000 preds=(64, 2)
epoch 1: loss=0.7697 preds=(64, 2)


## 読み取りどころ

- **`eval()` の付け忘れは例外を出さない。**静かに精度が数 % 下がるだけなので、原因の特定が遅れる。
- **③ は本番で必ず踏む。**リクエストを 1 件ずつ処理する API で `train()` のままだと、BatchNorm が `ValueError: Expected more than 1 value per channel` で落ちる。**バッチ推論では通るのに 1 件推論だけ落ちる**ので、負荷の低い時間帯にだけ出る不具合になりやすい。（Dropout はこの場合も例外を出さず、静かに出力を変えるだけなのでさらに厄介である。）
- `torch.no_grad()` は `eval()` とは別の話。**`eval()` は層の挙動、`no_grad()` は勾配計算の有無**を切り替える。両方必要である。